# Import libraries

In [1]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from pandas_gbq import to_gbq
import os

print('Libraries imported successfully')

Libraries imported successfully


In [2]:
# Set the environment variable for Google Cloud credentials
# Place the path in which the .json file is located.

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Users\roxan\AppData\Roaming\gcloud\application_default_credentials.json"

In [3]:
# Set your Google Cloud project ID and BigQuery dataset details

project_id = 'project-401f4646-3663-4125-aaa' # Edit with your project id
dataset_id = 'staging_db' # Modify the necessary schema name: staging_db, reporting_db etc.
table_id = 'stg_film' # Modify the necessary table name: stg_customer, stg_city etc.

# SQL Query

In [4]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Define your SQL query here
query = """
with base as (
  select *
  from `project-401f4646-3663-4125-aaa.pagila_productionpublic.film` 
  )

  , final as (
    select
          film_id
        , title as film_title
        , description as film_description
        , language_id as film_language_id
        , original_language_id as film_original_language_id
        , rental_duration as film_rental_duration
        , rental_rate as film_rental_rate
        , length as film_length
        , replacement_cost as film_replacement_cost
        , rating as film_rating
        , last_update as film_last_update
        , special_features as film_special_features
        , fulltext as film_fulltext
   FROM base
  )

  select * from final
"""

# Execute the query and store the result in a dataframe
df = client.query(query).to_dataframe()

# Explore some records
df.head()

C:\Users\roxan\PycharmProjects\Pagila Analytics Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,film_id,film_title,film_description,film_language_id,film_original_language_id,film_rental_duration,film_rental_rate,film_length,film_replacement_cost,film_rating,film_last_update,film_special_features,film_fulltext
0,960,WARS PLUTO,A Taut Reflection of a Teacher And a Database ...,1,<NA>,5,2.990000000,128,15.990000000,G,2022-09-10 16:46:03.905795+00:00,"[""Commentaries"",""Behind the Scenes""]",'administr':12 'chase':15 'databas':11 'desert...
1,497,KILL BROTHERHOOD,A Touching Display of a Hunter And a Secret Ag...,1,<NA>,4,0.990000000,54,15.990000000,G,2022-09-10 16:46:03.905795+00:00,"[""Trailers"",""Commentaries""]",'agent':12 'brotherhood':2 'display':5 'hunter...
2,653,PANIC CLUB,A Fanciful Display of a Teacher And a Crocodil...,1,<NA>,3,4.990000000,102,15.990000000,G,2022-09-10 16:46:03.905795+00:00,"[""Commentaries"",""Deleted Scenes""]",'baloon':19 'club':2 'crocodil':11 'display':5...
3,43,ATLANTIS CAUSE,A Thrilling Yarn of a Feminist And a Hunter wh...,1,<NA>,6,2.990000000,170,15.990000000,G,2022-09-10 16:46:03.905795+00:00,"[""Behind the Scenes""]",'atlanti':1 'caus':2 'feminist':8 'fight':14 '...
4,402,HARPER DYING,A Awe-Inspiring Reflection of a Woman And a Ca...,1,<NA>,3,0.990000000,52,15.990000000,G,2022-09-10 16:46:03.905795+00:00,"[""Trailers""]",'awe':5 'awe-inspir':4 'cat':13 'confront':16 ...


# Write to BigQuery

In [5]:
# Define the full table ID
full_table_id = f"{project_id}.{dataset_id}.{table_id}"

# Define table schema based on the project description

schema = [
    bigquery.SchemaField('film_id', 'INTEGER'),
    bigquery.SchemaField('film_title', 'STRING'),
    bigquery.SchemaField('film_description', 'STRING'),
    bigquery.SchemaField('film_language_id', 'INTEGER'),
    bigquery.SchemaField('film_original_language_id', 'INTEGER'),
    bigquery.SchemaField('film_rental_duration', 'INTEGER'),
    bigquery.SchemaField('film_rental_rate', 'NUMERIC'),
    bigquery.SchemaField('film_length', 'INTEGER'),
    bigquery.SchemaField('film_replacement_cost', 'NUMERIC'),
    bigquery.SchemaField('film_rating', 'STRING'),
    bigquery.SchemaField('film_last_update', 'DATETIME'),
    bigquery.SchemaField('film_special_features', 'STRING'),
    bigquery.SchemaField('film_fulltext', 'STRING'),
    ]

In [6]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Check if the table exists
def table_exists(client, full_table_id):
    try:
        client.get_table(full_table_id)
        return True
    except Exception:
        return False

# Write the dataframe to the table (overwrite if it exists, create if it doesn't)
if table_exists(client, full_table_id):
    # If the table exists, overwrite it
    destination_table = f"{dataset_id}.{table_id}"
    # Write the dataframe to the table (overwrite if it exists)
    to_gbq(df, destination_table, project_id=project_id, if_exists='replace')
    print(f"Table {full_table_id} exists. Overwritten.")
else:
    # If the table does not exist, create it
    job_config = bigquery.LoadJobConfig(schema=schema)
    job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
    job.result()  # Wait for the job to complete
    print(f"Table {full_table_id} did not exist. Created and data loaded.")

Table project-401f4646-3663-4125-aaa.staging_db.stg_film did not exist. Created and data loaded.


In [7]:
# Converting i.pynb file to .py python executable file. 

!python -m jupyter nbconvert stg_film.ipynb --to python --output-dir='../'

[NbConvertApp] Converting notebook stg_film.ipynb to python
[NbConvertApp] Writing 4010 bytes to ..\stg_film.py


In [8]:
!python -m pip install nbconvert


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!python -m pip install nbconvert -U


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
